# Stellar Spectral Classification with a Multi-Layer Perceptron

**Objective.** Build a neural network that classifies stars into
Morgan-Keenan (MK) spectral types -- O, B, A, F, G, K, M -- from their
optical spectra.

**What you will learn:**

| Topic | Section |
|---|---|
| Blackbody radiation and Planck's law | 1 |
| Atomic absorption lines in stellar atmospheres | 2 |
| The Morgan-Keenan classification system | 2 |
| Loading and exploring a spectral dataset | 3 |
| Data preprocessing for neural networks | 4 |
| Building and training an MLP in PyTorch | 5-6 |
| Evaluation: confusion matrix, per-class metrics | 7 |
| Gradient-based feature importance (which wavelengths matter?) | 8 |

**Prerequisites:** basic Python, introductory optics, no prior ML experience
required.

> **Note:** The dataset file `stellar_spectra.npz` should be uploaded to
> your Colab session (or placed in the working directory).  Your instructor
> will provide it.

In [ ]:
# ── Install & import ─────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import warnings, random, os

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device: {device}")

---
## 1. Blackbody Radiation and Planck's Law

A star's continuous spectrum is well approximated by a **blackbody** at its
effective temperature $T_{\mathrm{eff}}$.  Planck's law gives the spectral
radiance:

$$
B_\lambda(\lambda, T) \;=\; \frac{2hc^2}{\lambda^5}
\;\frac{1}{\exp\!\bigl(\tfrac{hc}{\lambda k_B T}\bigr) - 1}
$$

where

| Symbol | Meaning | Value |
|--------|---------|-------|
| $h$ | Planck constant | $6.626 \times 10^{-34}\;\mathrm{J\,s}$ |
| $c$ | Speed of light | $2.998 \times 10^{8}\;\mathrm{m/s}$ |
| $k_B$ | Boltzmann constant | $1.381 \times 10^{-23}\;\mathrm{J/K}$ |

**Wien's displacement law** tells us the peak wavelength shifts with
temperature: $\lambda_{\max} = b / T$, with $b \approx 2.898 \times 10^{-3}\;\mathrm{m\,K}$.

### Task 1.1

Implement Planck's law and plot normalised curves for the temperatures
of each MK spectral type.

In [ ]:
# ── Task 1.1: Implement Planck's law ────────────────────────────────────
h  = 6.62607015e-34   # J s
c  = 2.99792458e8     # m/s
kB = 1.380649e-23     # J/K

def planck(lam, T):
    '''Spectral radiance B_lambda(lambda, T) in W / (m^3 sr).

    Parameters
    ----------
    lam : array-like   Wavelength in metres.
    T   : float        Temperature in Kelvin.

    Returns
    -------
    B : ndarray   Spectral radiance.
    '''
    lam = np.asarray(lam, dtype=np.float64)
    # TODO: implement the Planck function (Eq. above)
    # Hint: use np.expm1(x) for exp(x) - 1 (numerically stable)
    raise NotImplementedError("Implement planck()")

# Representative temperatures for each MK type
temperatures = {
    "O  (40 000 K)": 40000,
    "B  (20 000 K)": 20000,
    "A  (9 000 K)":  9000,
    "F  (7 000 K)":  7000,
    "G  (5 800 K)":  5800,
    "K  (4 500 K)":  4500,
    "M  (3 200 K)":  3200,
}

lam_m = np.linspace(100e-9, 2500e-9, 2000)      # 100 nm to 2500 nm
lam_nm = lam_m * 1e9

# TODO: plot normalised Planck curves for each temperature
# - Use different colours for each type
# - Mark the visible band (380-700 nm) with axvspan
# - Add labels and a legend
fig, ax = plt.subplots(figsize=(10, 5))
# YOUR CODE HERE

plt.tight_layout(); plt.show()

---
## 2. Absorption Lines and the Morgan-Keenan System

A stellar atmosphere absorbs photons at specific wavelengths determined by
the atomic and molecular species present and their excitation states.  The
**depth** of each absorption line depends on temperature and surface
gravity.

### Key spectral features

| Feature | Wavelength (nm) | Strong in types |
|---------|---------------:|-----------------|
| Ca II K | 393.4 | F, G, K |
| Ca II H | 396.8 | F, G, K |
| H-$\delta$ | 410.2 | A, B |
| H-$\gamma$ | 434.0 | A, B |
| H-$\beta$ | 486.1 | A, B (strongest in A) |
| Mg I b | 518.4 | G, K |
| Fe I | 527.0 | G, K |
| Na D | 589.3 | K, M |
| He I | 587.6 | O, B |
| H-$\alpha$ | 656.3 | A (max), weaker in B, F |
| TiO band | 705-720 | M (dominant) |

### The MK classification

The **Morgan-Keenan** system assigns each star a **spectral type**
(O, B, A, F, G, K, M) based on these features.  The sequence runs from
hottest (O) to coolest (M), remembered by the mnemonic:

> *Oh Be A Fine Girl/Guy, Kiss Me*

---
## 3. Loading and Exploring the Dataset

The dataset file `stellar_spectra.npz` contains 3,500 synthetic stellar
spectra (500 per MK type) sampled on a uniform wavelength grid from
380 nm to 900 nm at 0.5 nm resolution.

Contents of the `.npz` file:
- `spectra`: shape `(3500, 1040)` -- flux values
- `labels`: shape `(3500,)` -- integer class labels (0=O, ..., 6=M)
- `wavelengths`: shape `(1040,)` -- wavelength grid in nm
- `class_names`: `['O', 'B', 'A', 'F', 'G', 'K', 'M']`
- `temperatures`: shape `(3500,)` -- effective temperature for each star

### Task 3.2: Visualise example spectra

Plot a few spectra from each spectral type.  Use the wavelength grid as
the x-axis.

---
## 4. Data Preprocessing

Before feeding spectra into a neural network we apply two standard steps:

1. **Standardisation.**  Each spectrum is normalised to zero mean and unit
   variance.  This removes the Planck slope so the network focuses on
   absorption features.

2. **Train / validation / test split.**  We use 70% / 15% / 15%.

> **Why standardise?** Neural networks converge faster when input features
> have similar scales.  Without standardisation the network would waste
> capacity learning the Planck slope, which is already encoded in the class
> label.

In [ ]:
# ── Task 4.1: Standardise and split ─────────────────────────────────────

# TODO: standardise each spectrum to zero mean, unit variance
# Hint: compute mean and std per spectrum (axis=1, keepdims=True)
# X_norm = ...


# Convert to tensors
X_tensor = torch.tensor(X_norm)
y_tensor = torch.tensor(y_all)

# TODO: split into train (70%), val (15%), test (15%) using random_split
# dataset = TensorDataset(X_tensor, y_tensor)
# n_total = len(dataset)
# n_train = ...
# n_val   = ...
# n_test  = ...
# train_ds, val_ds, test_ds = random_split(...)


# Create DataLoaders
BATCH = 64
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH)
test_dl  = DataLoader(test_ds,  batch_size=BATCH)

print(f"train {n_train} | val {n_val} | test {n_test}")
xb, yb = next(iter(train_dl))
print(f"Batch shapes: X {xb.shape}, y {yb.shape}")

---
## 5. The Multi-Layer Perceptron (MLP)

An MLP is a sequence of **fully-connected (linear) layers** separated by
non-linear **activation functions**.  For an input vector $\mathbf{x} \in
\mathbb{R}^d$:

$$
\mathbf{h}_1 = \sigma(W_1\,\mathbf{x} + \mathbf{b}_1), \quad
\mathbf{h}_2 = \sigma(W_2\,\mathbf{h}_1 + \mathbf{b}_2), \quad
\hat{\mathbf{y}} = W_3\,\mathbf{h}_2 + \mathbf{b}_3
$$

where $\sigma$ is the activation function (we use **ReLU**: $\sigma(z) =
\max(0, z)$).  The output $\hat{\mathbf{y}} \in \mathbb{R}^C$ contains
raw **logits** for each of the $C = 7$ classes.

### Task 5.1

Define an MLP with:
- 3 hidden layers of sizes 256, 128, 64
- BatchNorm + ReLU + Dropout(0.3) after each hidden layer
- Output layer: 7 neurons (one per class)

In [ ]:
# ── Task 5.1: Define the MLP ────────────────────────────────────────────

class StellarMLP(nn.Module):
    def __init__(self, n_features, n_classes, hidden=(256, 128, 64), drop=0.3):
        super().__init__()
        # TODO: build the network as nn.Sequential
        # For each hidden size: Linear -> BatchNorm1d -> ReLU -> Dropout
        # Final layer: Linear(last_hidden, n_classes)
        raise NotImplementedError("Build the MLP")

    def forward(self, x):
        return self.net(x)

model = StellarMLP(NUM_FEATURES, NUM_CLASSES).to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {total_params:,}")

---
## 6. Training the MLP

We use:
- **Cross-entropy loss**: $\mathcal{L} = -\log \hat{p}_{\text{true}}$
- **Adam optimiser** with learning rate $10^{-3}$
  stagnates for 5 epochs

### Task 6.1

Implement the training loop.  For each epoch:
1. Run all training batches (forward pass, loss, backward, optimiser step)
2. Evaluate on the validation set (no gradients needed)
3. Record loss and accuracy for both sets
4. Step the scheduler with the validation loss

In [ ]:
# ── Task 6.1: Training loop ─────────────────────────────────────────────
EPOCHS = 60
LR = 1e-3

criterion = nn.CrossEntropyLoss()
optimiser = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimiser, mode="min", factor=0.5, patience=5
)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

def run_epoch(loader, train=True):
    '''Run one epoch. Returns (avg_loss, accuracy).'''
    model.train(train)
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            # TODO:
            # 1. Forward pass: logits = model(xb)
            # 2. Compute loss
            # 3. If training: zero_grad, backward, step
            # 4. Accumulate loss and count correct predictions
            raise NotImplementedError("Implement the epoch loop")
    return total_loss / total, correct / total

# TODO: training loop over EPOCHS
# For each epoch: call run_epoch for train and val,
# step the scheduler, record history, print every 10 epochs
raise NotImplementedError("Implement the training loop")

In [ ]:
# ── Task 6.2: Plot training curves ──────────────────────────────────────

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# TODO: plot train/val loss on ax1, train/val accuracy on ax2
# Add labels, legend, and titles

plt.tight_layout(); plt.show()

---
## 7. Evaluation on the Test Set

### Task 7.1

Evaluate the trained model on the test set.  Compute:
- Overall accuracy
- Confusion matrix
- Per-class precision, recall, F1 (use `classification_report`)

In [ ]:
# ── Task 7.1: Test evaluation ───────────────────────────────────────────
model.eval()
all_preds, all_labels = [], []

# TODO: loop over test_dl, collect predictions and true labels
# Hint: use torch.no_grad(), model(xb).argmax(1)
raise NotImplementedError("Evaluate on the test set")

all_preds  = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()
test_acc = (all_preds == all_labels).mean()
print(f"TEST accuracy: {test_acc:.4f}\n")
print(classification_report(all_labels, all_preds,
                            target_names=list(CLASS_NAMES), digits=3))

In [ ]:
# ── Task 7.2: Confusion matrix ──────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)

# TODO: plot the confusion matrix as a colour-coded grid
# - Use plt.imshow with cmap="Blues"
# - Add text annotations in each cell
# - Label axes with class names
fig, ax = plt.subplots(figsize=(7, 6))
# YOUR CODE HERE

plt.tight_layout(); plt.show()

### Questions

1. Which pair of spectral types is hardest to distinguish?  What physical
   reason explains this?


---
## 9. Bonus Exercises

**Exercise A -- Architecture search.**
Try different hidden layer sizes.  Plot test accuracy vs. total parameter
count.

**Exercise B -- Noise robustness.**
Add extra Gaussian noise to the test spectra and measure how accuracy
degrades.

**Exercise C -- PCA dimensionality reduction.**
Apply PCA to the spectra.  How many components capture 99% of the variance?
Train the MLP on PCA features -- does accuracy change?  Is training faster?

In [ ]:
# ── Starter code for Exercise C (PCA) ───────────────────────────────────
from sklearn.decomposition import PCA

# TODO:
# 1. Fit PCA on training data only (X_norm[train_ds.indices])
# 2. Plot cumulative explained variance
# 3. Find n_components for 99% variance
# 4. Transform data and retrain the MLP with fewer input features

---
*Notebook prepared for optics / astrophysics students learning machine
learning.*